In [ ]:
########################## split dataset ###############################
import pandas as pd
import numpy as np

# -------------------------
# Config
# -------------------------
INPUT_CSV = "/data/colon_cancer/Classifier/ColonCancer/labels_cleaned.csv"
OUTPUT_CSV = "/data/colon_cancer/Classifier/ColonCancer/splits_cleaned.csv"

N_TRAIN = 600
N_VAL   = 118
N_TEST  = 77

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# -------------------------
# Load data
# -------------------------
df = pd.read_csv(INPUT_CSV)

assert len(df) == N_TRAIN + N_VAL + N_TEST, "Split sizes do not sum to dataset size!"

# -------------------------
# Compute class proportions
# -------------------------
class_counts = df["target"].value_counts().sort_index()
total = len(df)

class_ratios = class_counts / total

# Samples per class per split
def split_counts(n_total):
    counts = (class_ratios * n_total).round().astype(int)
    # fix rounding errors
    diff = n_total - counts.sum()
    if diff != 0:
        counts.iloc[0] += diff
    return counts

train_counts = split_counts(N_TRAIN)
val_counts   = split_counts(N_VAL)
test_counts  = split_counts(N_TEST)

# -------------------------
# Perform stratified split
# -------------------------
df["split"] = None
remaining_idx = []

for cls in class_counts.index:
    cls_df = df[df["target"] == cls].sample(frac=1, random_state=RANDOM_SEED)

    n_train = train_counts[cls]
    n_val   = val_counts[cls]
    n_test  = test_counts[cls]

    train_idx = cls_df.iloc[:n_train].index
    val_idx   = cls_df.iloc[n_train:n_train + n_val].index
    test_idx  = cls_df.iloc[n_train + n_val:n_train + n_val + n_test].index

    df.loc[train_idx, "split"] = "train"
    df.loc[val_idx, "split"]   = "val"
    df.loc[test_idx, "split"]  = "test"

# -------------------------
# Sanity checks
# -------------------------
print("\nSplit sizes:")
print(df["split"].value_counts())

print("\nClass balance per split:")
print(df.groupby(["split", "target"]).size().unstack())

assert df["split"].isna().sum() == 0, "Some samples were not assigned a split!"

# -------------------------
# Save
# -------------------------
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved splits to {OUTPUT_CSV}")


In [ ]:
##################### move test samples ###############################import os
import shutil
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Paths (EDIT THESE)
# --------------------------------------------------
splits_csv = "/data/colon_cancer/Classifier/ColonCancer/splits_cleaned.csv"

images_src = Path("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/imagesTr")
labels_src = Path("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labelsTr")

images_dst = Path("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/imagesTs")
labels_dst = Path("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labelsTs")

# Create destination folders
images_dst.mkdir(parents=True, exist_ok=True)
labels_dst.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Load CSV
# --------------------------------------------------
df = pd.read_csv(splits_csv)

# Normalize columns
df["UID"] = df["UID"].astype(str)
df["split"] = df["split"].str.lower()

# Select test set
test_uids = df[df["split"] == "test"]["UID"].tolist()

print(f"Found {len(test_uids)} test cases")

# --------------------------------------------------
# Move files
# --------------------------------------------------
moved_images = 0
moved_labels = 0

for uid in test_uids:
    img_name = f"{uid}_0000.nii.gz"
    lbl_name = f"{uid}.nii.gz"

    img_src = images_src / img_name
    lbl_src = labels_src / lbl_name

    img_dst = images_dst / img_name
    lbl_dst = labels_dst / lbl_name

    # Move image
    if img_src.exists():
        shutil.move(str(img_src), str(img_dst))
        moved_images += 1
        print(f"✓ Moved image: {img_name}")
    else:
        print(f"⚠ Image missing: {img_name}")

    # Move label
    if lbl_src.exists():
        shutil.move(str(lbl_src), str(lbl_dst))
        moved_labels += 1
        print(f"✓ Moved label: {lbl_name}")
    else:
        print(f"⚠ Label missing: {lbl_name}")

# --------------------------------------------------
# Summary
# --------------------------------------------------
print("\n=== Summary ===")
print(f"Images moved: {moved_images}")
print(f"Labels moved: {moved_labels}")
print("Done ✅")